### 工具调用
- 绑定工具
- 调用模型,模型自主选择是否调用工具
- 工具调用返回结果并追加进入消息列表
- 调用模型,直到输出最终结果


In [3]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_classic.agents.chat.prompt import HUMAN_MESSAGE
from scripts.regsetup import description

load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    api_key= DEEPSEEK_API_KEY,
    base_url= DEEPSEEK_BASE_URL,
    model= "deepseek-v4-flash",
)




使用@tool装饰器的多工具调用
- 举例:

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from pydantic import BaseModel,Field
from rich import print as rprint

#args_schema参数类
class WeatherArgs(BaseModel):
    if_forecast: bool = Field(default=False, description="是否查询未来24小时内天气")
    city: str = Field(description="城市名称")

#tool装饰器,description工具描述,args_schema参数类,name_or_recall重命名
@tool(description="获取城市的天气",args_schema=WeatherArgs)
def get_weather(city: str,if_forecast: bool = False) -> str:
    res = f"{city}的天气晴朗"
    if if_forecast:
        res += "\n未来24小时内天气有小雨"
    return res


class NewaArgs(BaseModel):
    date: str = Field(description="日期")

@tool(description="获取指定日期的新闻",args_schema=NewaArgs)
def get_news(date: str) -> list:
    news = [
    "浙江锚定高质量发展建设共同富裕示范区，以'千万工程'牵引城乡融合发展取得新成效。",
    "国产大飞机C919正式开启国际定期商业航线运营，由国航执飞的CA723航班降落乌兰巴托机场。",
    "北京市商务局等6部门联合印发措施，促进餐饮和食品领域老字号传承发展。"
        ]
    return news

#维护消息列表作为记忆
messages: list[BaseModel] = [HumanMessage("北京今天,明天天气如何,今天的新闻有哪些")]

model_with_tools = model.bind_tools([get_weather,get_news])

rprint(convert_to_openai_tool(get_weather))

while True:
    response = model_with_tools.invoke(messages)
    messages.append(response)
    if not response.tool_calls:
        break
    #遍历工具调用,根据工具名称调用工具,并追加进入消息列表
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_weather":
            tool_res = get_weather.invoke(tool_call)#工具名,返回为ToolMessage 类型
            messages.append(tool_res)
        elif tool_call["name"] == "get_news":
            tool_res = get_news.invoke(tool_call)#工具名,返回为ToolMessage 类型
            messages.append(tool_res)
        else :
            raise ValueError(f"未知工具调用:{tool_call['name']}")

for mes in messages:
    mes.pretty_print()


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {
                'if_forecast': {'default': False, 'description': '是否查询未来24小时内天气', 'type': 'boolean'},
                'city': {'description': '城市名称', 'type': 'string'}
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

================================ Human Message =================================

北京今天,明天天气如何,今天的新闻有哪些
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_TrON7HKrncYvYr6oQjPu7051)
 Call ID: call_00_TrON7HKrncYvYr6oQjPu7051
  Args:
    city: 北京
    if_forecast: True
  get_news (call_01_ytBdOXzbY1TUX2j9jznO4171)
 Call ID: call_01_ytBdOXzbY1TUX2j9jznO4171
  Args:
    date: 2025-06-01
================================= Tool Message =================================
Name: get_weather

北京的天气晴朗
未来24小时内天气有小雨
================================= Tool Message =================================
Name: get_news

["浙江锚定高质量发展建设共同富裕示范区，以'千万工程'牵引城乡融合发展取得新成效。", '国产大飞机C919正式开启国际定期商业航线运营，由国航执飞的CA723航班降落乌兰巴托机场。', '北京市商务局等6部门联合印发措施，促进餐饮和食品领域老字号传承发展。']
================================== Ai Message ==================================

为您查询到以下信息：

## ☀️ 北京天气

- **今天**：晴朗 ☀️
- **明天**：未来24小时内将有小雨 🌦️

## 📰 今日新闻

1. **浙江锚定高质量发展建设共同富裕示范区**，以"千万工程"牵引城乡融合发展取得

## @tool常用参数解释
- description:描述工具的功能
- args_schema:参数类,绑定继承BaseModel的参数类
- name_or_recall:工具重命名,默认为函数名,@tool("new_get_weather")不带参数时默认重命名
- parse_docstring:是否解析函数文档字符串(谷歌格式),默认False,开启后优先以@tool装饰器的参数为准
### parse_docstring举例
```python
@tool(parse_docstring=True)
def search_flights(origin: str, destination: str, date: str) -> str:
    """
    搜索航班信息

    Args:
    origin: 出发城市，如"北京"
    destination: 目的地城市，如"上海"
    date: 出发日期，格式 YYYY-MM-DD

    Returns:
    可用航班的 JSON 列表
    """
```


### 实战经验总结
1.职责单一原则,每个工具只负责一个功能,避免工具功能复杂
- 错误描述:"""do everything"""
2.工具描述要清晰,参数解释到位
3.处理工具调用失败,3层处理
- 工具代码错误,工具内部try-except,捕获异常,返回错误信息
- agent级别重试,系统提示词注入,例:prompt="如果工具失败，尝试使用其他方法解决问题。"
- 网络请求,外部工具调用,@retry(stop=stop_after_attempt(3))
    - 解释:配置重试规则：如果失败，最多尝试 3 次（即第 1 次正常调用 + 2 次重试）
    - 位置:调用大模型函数上
4.str返回,Langchain@tool生态强烈建议返回str类型
- LLM输入本身是str类型,返回str类型,避免类型转换
- 防止dict等类型让大模型胡思乱想
5.异步与同步
- 同步:cpu密集型任务,任务执行时间短,场景简单
- 异步:IO密集型任务（API 调用、数据库、文件操作）,任务执行时间长,场景复杂
